Gemini said (about adapting this env to SARSA):

A tabular Q-table requires discrete, finite states and actions. Your action space is naturally discrete, but your observation space is continuous and structured as a dictionary, making a direct Q-table representation impossible without transformation or approximation.

---

### 1. Action Space Size

The action space is a `MultiDiscrete([11, 11, 11])`, representing three independent choices of 11 discrete levels each:

$$\text{actions\_size} = 11 \times 11 \times 11 = 1{,}331$$

To map an action tuple $(a_1, a_2, a_3)$ to a single scalar index in $[0, 1330]$:

```python
import numpy as np

def action_to_index(action):
    # action is an array-like [a0, a1, a2] where each ai in [0, 10]
    return np.ravel_multi_index(action, (11, 11, 11))

def index_to_action(idx):
    return np.unravel_index(idx, (11, 11, 11))

```

---

### 2. Observation Space Size (The Problem)

Your observation space contains **38 continuous values** in total:

* `inventory`: 3 values
* `arrival_pipeline`: $3 \times 4 = 12$ values
* `demand_history`: $7 \times 3 = 21$ values
* `day`: 1 value
* `capacity_utilisation`: 1 value

Even if you treat each variable as an integer without discretization:

* $1{,}001^3 \approx 10^9$ states for `inventory` alone.
* The total state space exceeds $10^{100}$, which cannot fit into memory as a lookup table.

---

### 3. How to Make It Work

#### Option A: Discretization (Bucketing / Binning)

Reduce the observation to a compact set of bins. Because 38 dimensions will still suffer from the curse of dimensionality, you must heavily summarize the features first:

1. **Feature Extraction:** Aggregate high-dimensional fields into small discrete metrics:
* Total or mean pipeline stock (e.g., 5 bins: Very Low, Low, Med, High, Very High)
* Average recent demand (e.g., 5 bins)
* Inventory levels per product (e.g., 5 bins each $\rightarrow 5^3 = 125$)
* Day (e.g., current day modulo 7, or broad buckets)


2. **Compute Total States:**
If your reduced summary produces a discrete tuple of sizes $(S_1, S_2, \dots, S_k)$:

$$\text{observations\_size} = \prod_{i=1}^{k} S_i$$


3. **Table Allocation:**
```python
num_states = 50_000  # Example target state count after bucketing
num_actions = 11 * 11 * 11
q_table = np.zeros((num_states, num_actions), dtype=np.float32)

```



#### Option B: Function Approximation (Recommended)

Because state spaces with high-dimensional demand and pipeline histories resist direct tabular mapping, standard practice for this environment design is to use **Deep Q-Networks (DQN)** or **Actor-Critic (PPO / A2C)**:

1. Flatten the observation dictionary into a 38-dimensional vector using `gymnasium.spaces.utils.flatten`.
2. Feed the 38-dimensional vector into a neural network that outputs 1,331 Q-values (or three heads of 11 values using Branching Dueling Q-Networks to handle multi-discrete actions cleanly).

```python
import gymnasium as gym

# Flatten the observation dict into a 1D float vector of shape (38,)
flat_obs = gym.spaces.utils.flatten(self.observation_space, obs)

```

Source: Tutorial Notebook: RL_Gymnasium_MC_SARSA_Tutorial.ipynb

In [ ]:
from copy import deepcopy
from pathlib import Path

from tqdm import trange
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from industrial_inventory_env import (
    IndustrialInventoryEnv,
    generate_student_config,
    public_config_summary,
)

np.set_printoptions(suppress=True)

In [2]:
ROLL_NUMBER = "DA25M622"

student_config = generate_student_config(ROLL_NUMBER)
config_summary = public_config_summary(student_config)

print("Assigned configuration generated successfully.")
for key, value in config_summary.items():
    print(f"{key}: {value}")

Assigned configuration generated successfully.
project_version: IITM-6002W-RL-Inventory-2026-v1
roll_number: DA25M622
variant_id: V022
config_fingerprint: 7d77fc79debf206a
demand_multiplier_profile: [1.0, 1.1, 0.9]
initial_inventory_profile: [110, 100, 90]
lead_time_delay_profile: [0.1, 0.0, 0.05]


In [3]:
env = IndustrialInventoryEnv(
    student_config=student_config,
    scenario_mode="random",
    domain_randomization=True,
)

observation, info = env.reset(seed=2026)

print("Action space:", env.action_space)
print("Observation space:", env.observation_space)
print("Variant:", info["variant_id"])
print("Scenario components:", info["episode_parameters"]["scenario_components"])
print("Episode demand multipliers:", info["episode_parameters"]["demand_multipliers"])
print("Episode initial inventory:", info["episode_parameters"]["initial_inventory"])
print("Episode delay probabilities:", info["episode_parameters"]["delay_probabilities"])

Action space: MultiDiscrete([11 11 11])
Observation space: Dict('arrival_pipeline': Box(0, 10000, (3, 4), int32), 'capacity_utilisation': Box(0.0, 1.0, (1,), float32), 'day': Box(0, 50, (1,), int32), 'demand_history': Box(0, 10000, (7, 3), int32), 'inventory': Box(0, 1000, (3,), int32))
Variant: V022
Scenario components: ['seasonal', 'trend']
Episode demand multipliers: [1.05, 1.05, 0.85]
Episode initial inventory: [110, 100, 90]
Episode delay probabilities: [0.08, 0.0, 0.05]


In [4]:
for key, value in observation.items():
    print(f"{key:22s} shape={value.shape}, dtype={value.dtype}")
    print(value)
    print()

inventory              shape=(3,), dtype=int32
[110 100  90]

arrival_pipeline       shape=(3, 4), dtype=int32
[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]

demand_history         shape=(7, 3), dtype=int32
[[0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]]

day                    shape=(1,), dtype=int32
[0]

capacity_utilisation   shape=(1,), dtype=float32
[0.655]



In [6]:
action_indices = env.quantities_to_action_indices(quantities=[40, 20, 0])
action_indices


array([4, 2, 0])

In [7]:
next_observation, reward, terminated, truncated, step_info = env.step(action_indices)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("Demand:", step_info["demand"])
print("Daily costs:", step_info["costs"])
print("Next inventory:", next_observation["inventory"])

Reward: -27.425
Terminated: False
Truncated: False
Demand: [34 21 21]
Daily costs: {'holding': 2462.5, 'stockout': 0.0, 'ordering': 280.0, 'discarding': 0.0, 'daily_total': 2742.5, 'episode_total': 2742.5}
Next inventory: [76 79 69]


In [ ]:
# RL training

def epsilon_greedy_action(Q, state, n_actions, epsilon, rng):
    '''Sample A_t from an epsilon-greedy policy based on Q(S_t, .).'''
    if rng.random() < epsilon:
        return int(rng.integers(n_actions))  # exploration

    action_values = Q[state]
    greedy_actions = np.flatnonzero(action_values == action_values.max())
    return int(rng.choice(greedy_actions))  # random tie-breaking

num_episodes=500
alpha=0.5
gamma=1.0
epsilon=0.10
seed = 2026
env = IndustrialInventoryEnv(
    student_config=student_config,
    scenario_mode="random",
    domain_randomization=True,
)
observation, info = env.reset(seed=seed)
env.action_space.seed(seed)           # NOTE: Unsure if needed, guess it can't hurt
rng = np.random.default_rng(seed)

Q = np.zeros((env.observation_space.n, env.action_space.n), dtype=float)
episode_returns = np.zeros(num_episodes, dtype=float)
episode_lengths = np.zeros(num_episodes, dtype=int)

for episode in trange(num_episodes):
    state, info = env.reset() # continues from the seeded random sequence
    action = epsilon_greedy_action(Q, state, env.action_space.n, epsilon, rng)

    while True:
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        episode_returns[episode] += reward
        episode_lengths[episode] = step + 1

        if done:
            td_target = reward  # no Q term after a terminal state
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error
            break

        # A_(t+1) comes from the SAME epsilon-greedy behaviour policy.
        next_action = epsilon_greedy_action(
            Q, next_state, env.action_space.n, epsilon, rng
        )
        td_target = reward + gamma * Q[next_state, next_action] # needed to compute td_error
        td_error = td_target - Q[state, action]
        Q[state, action] += alpha * td_error # thus values are being stored

        state, action = next_state, next_action

env.close()

print("Training episodes       :", len(episode_returns))
print("Mean return, first 50   :", episode_returns[:50].mean().round(1))
print("Mean return, last 50    :", episode_returns[-50:].mean().round(1))
print("Mean length, last 50    :", episode_lengths[-50:].mean().round(1))


AttributeError: 'Dict' object has no attribute 'n'